[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrjon/BayesFlowTutorial/blob/main/03_Diffusion_Guidance_Exercise.ipynb)

# BayesFlow 3 — Diffusion Models & Custom Guidance

**Difficulty:** 🔴 Advanced  ·  **Est. time:** ~90 min

> 🎯 **What you will learn**
> - Train a **diffusion-model** posterior estimator for a multimodal, non-identifiable inverse problem (inverse kinematics).
> - Understand diffusion models as **score learning** + reverse SDE/ODE sampling, and how flow matching / consistency models relate.
> - **Write your own guidance constraint** from scratch and steer posterior sampling *after* training via `guidance_kwargs` — the core, open-ended task.

---

> ✏️ **How this notebook works.** Look for **`TODO`** markers. Each one asks you to fill in a small piece of code (marked with `...` or a `NotImplementedError`). Hints and documentation links sit right next to every task. The matching **solution notebook** contains one possible answer — try it yourself first!

In [ ]:
# ▶️  RUN THIS FIRST.  Installs this tutorial project and all its dependencies
#     (BayesFlow, JAX, Pyro, ...) from GitHub, as declared in pyproject.toml (~1-2 min).
import sys, subprocess

print("Installing the tutorial and its dependencies — this takes ~1-2 min ...")
_res = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/arrjon/BayesFlowTutorial.git"],
    capture_output=True, text=True,
)
if _res.returncode != 0:
    # A real failure — show pip's output so it can be debugged.
    print(_res.stdout); print(_res.stderr)
    raise SystemExit("Install failed — see the pip output above.")

# NOTE: pip may internally warn that google-colab pins pandas==2.2.2 while BayesFlow
# needs pandas>=2.3.3. That warning is harmless and unavoidable (the tutorial doesn't use
# google-colab's pandas features); we install quietly so it doesn't clutter the output.
print("✅ Setup complete.")
print("   If the NEXT cell raises a numpy/pandas import error, click "
      "Runtime ▸ Restart session and re-run.")

In [ ]:
import os
# BayesFlow runs on JAX, PyTorch or TensorFlow. We pick JAX here (installed on Colab).
os.environ["KERAS_BACKEND"] = "jax"
import bayesflow as bf

> 📚 **Where to read more**
> - BayesFlow website & docs: <https://bayesflow.org>
> - Example gallery (great for copy-paste starting points): <https://bayesflow.org/main/examples.html>
> - API reference: <https://bayesflow.org/main/api/bayesflow.html>

## The problem: inverse kinematics

A 3-segment planar robot arm has an unknown configuration $\boldsymbol\theta = (h, \alpha_1, \alpha_2, \alpha_3)$: a base **height** $h$ and three **joint angles**. Given an observed end-effector position $\mathbf{x} \in \mathbb{R}^2$, we want the posterior $p(\boldsymbol\theta \mid \mathbf{x})$.

This is deliberately **non-identifiable**: many arm configurations reach (almost) the same end position. So the posterior is **multimodal** — a perfect stress test for flexible generative estimators like diffusion models.

Reference: Kruse et al., *Benchmarking Invertible Architectures* — <https://arxiv.org/pdf/2101.10763.pdf>

### SBI recap
We can't evaluate the likelihood, but we can **simulate** pairs $(\boldsymbol\theta_i, \mathbf{x}_i)$. Training a conditional generative network on these pairs yields an amortized approximation $q_\phi(\boldsymbol\theta \mid \mathbf{x})$ — instant posteriors for any new $\mathbf{x}$. (If you haven't done the linear-regression and SIR notebooks yet, start there.)

### Plotting helper (run me, no need to read)

The cell below defines `InverseKinematicsModel`, a small utility that draws robot-arm configurations from posterior samples. It's pure plotting — just run it and move on.

In [ ]:
import keras
from collections import OrderedDict
from matplotlib import cm, patches
import matplotlib.pyplot as plt
import numpy as np

import scipy
from sklearn.cluster import MeanShift
from sklearn.neighbors import KernelDensity


class InverseKinematicsModel:
    n_parameters = 4
    n_observations = 2
    name = "inverse-kinematics"

    def __init__(self, lens=[0.5, 0.5, 1.0], sigmas=[0.25, 0.5, 0.5, 0.5], linecolors=["gray"] * 3):
        self.name = "inverse-kinematics"
        self.lens = np.array(lens)
        self.sigmas = np.array(sigmas)
        self.rangex = (-0.35, 2.25)
        self.rangey = (-1.3, 1.3)

        cmap = cm.tab20c
        self.colors = [[cmap(4 * c_index), cmap(4 * c_index + 1), cmap(4 * c_index + 2)] for c_index in range(5)][-1]
        self.linecolors = linecolors

        self.prior_reference = OrderedDict(
            theta_1=scipy.stats.norm(0, sigmas[0]),
            theta_2=scipy.stats.norm(0, sigmas[1]),
            theta_3=scipy.stats.norm(0, sigmas[2]),
            theta_4=scipy.stats.norm(0, sigmas[3]),
        )

    def sample_prior(self, N):
        return np.random.randn(N, 4) * self.sigmas

    @staticmethod
    def segment_points(p_, length, angle):
        p = np.array(p_)
        angle = np.array(angle)
        p[:, 0] += length * np.cos(angle)
        p[:, 1] += length * np.sin(angle)
        return p_, p

    def forward_process(self, x):
        start = np.stack([np.zeros((x.shape[0])), x[:, 0]], axis=1)
        _, x1 = self.segment_points(start, self.lens[0], x[:, 1])
        _, x2 = self.segment_points(x1, self.lens[1], x[:, 1] + x[:, 2])
        _, y = self.segment_points(x2, self.lens[2], x[:, 1] + x[:, 2] + x[:, 3])
        return y

    @staticmethod
    def find_MAP(x):
        mean_shift = MeanShift()
        mean_shift.fit(x)
        centers = mean_shift.cluster_centers_
        kde = KernelDensity(kernel="gaussian", bandwidth=0.1).fit(x)

        best_center = (None, -np.inf)
        dens = kde.score_samples(centers)
        for c, d in zip(centers, dens):
            if d > best_center[1]:
                best_center = (c.copy(), d)

        dist_to_best = np.sum((x - best_center[0]) ** 2, axis=1)
        return np.argmin(dist_to_best)

    @staticmethod
    def arcarrow(
        start,
        target,
        dist=0.3,
        open_angle=150,
        kw=dict(arrowstyle="<->, head_width=1, head_length=2", ec="black", lw=0.5),
    ):
        direction = target - start
        angle = np.arctan2(direction[1], direction[0])

        angle1 = angle - np.radians(open_angle / 2)
        x1 = start[0] + dist * np.cos(angle1)
        y1 = start[1] + dist * np.sin(angle1)
        angle2 = angle + np.radians(open_angle / 2)
        x2 = start[0] + dist * np.cos(angle2)
        y2 = start[1] + dist * np.sin(angle2)

        plt.gca().add_patch(patches.FancyArrowPatch((x1, y1), (x2, y2), connectionstyle="arc3, rad=.6", **kw))

    def draw_isolines(self, samples, color, filter_width, ax=None):
        if not filter_width > 0:
            return

        x = np.array(samples)

        starting_pos = np.zeros((x.shape[0], 2))
        starting_pos[:, 1] = x[:, 0]

        x0, x1 = self.segment_points(starting_pos, self.lens[0], x[:, 1])
        x1, x2 = self.segment_points(x1, self.lens[1], x[:, 1] + x[:, 2])
        x2, y = self.segment_points(x2, self.lens[2], x[:, 1] + x[:, 2] + x[:, 3])

        hist, xbins, ybins = np.histogram2d(y[:, 0], y[:, 1], bins=600, range=[self.rangex, self.rangey], density=True)
        hist = scipy.ndimage.gaussian_filter(hist, filter_width)

        percentile = 0.03 * np.sum(hist)
        for q in np.logspace(-99, np.log10(np.max(hist)), 8000, endpoint=True):
            if np.sum(hist[hist < q]) > percentile:
                break
        else:
            q = 1.0

        X, Y = np.meshgrid(0.5 * (xbins[:-1] + xbins[1:]), 0.5 * (ybins[:-1] + ybins[1:]))

        if ax is None:
            plt.contour(X, Y, hist.T, [q], colors=color, linewidths=0.7, zorder=3)
        else:
            ax.contour(X, Y, hist.T, [q], colors=color, linewidths=0.7, zorder=3)

    @staticmethod
    def init_plot():
        return plt.figure(figsize=(8, 8))

    def update_plot_ax(
        self,
        ax,
        x,
        y_target,
        exemplar=None,
        arrows=False,
        target_label=False,
        vline_color="black",
        exemplar_color="#e6e7eb",
        cross_color="maroon",
    ):
        x = np.array(x)  # [:4000, :]
        if exemplar is None:
            exemplar = self.find_MAP(x)

        starting_pos = np.zeros((x.shape[0], 2))
        starting_pos[:, 1] = x[:, 0]
        x0, x1 = self.segment_points(starting_pos, self.lens[0], x[:, 1])
        x1, x2 = self.segment_points(x1, self.lens[1], x[:, 1] + x[:, 2])
        x2, x3 = self.segment_points(x2, self.lens[2], x[:, 1] + x[:, 2] + x[:, 3])

        ax.axvline(x=0, c=vline_color, linewidth=1, alpha=0.8)

        if not arrows:
            l_cross = 0.6
            ax.plot(
                [y_target[0] - l_cross, y_target[0] + l_cross],
                [y_target[1], y_target[1]],
                ls="-",
                c=cross_color,
                linewidth=0.8,
                alpha=1.0,
                rasterized=True,
            )  # , zorder=-1)
            ax.plot(
                [y_target[0], y_target[0]],
                [y_target[1] - l_cross, y_target[1] + l_cross],
                ls="-",
                c=cross_color,
                linewidth=0.8,
                alpha=1.0,
                rasterized=True,
            )  # , zorder=-1)
            if target_label:
                ax.text(
                    y_target[0] + 0.15,
                    y_target[1] + 0.15,
                    target_label,
                    ha="left",
                    va="bottom",
                    color="magenta",
                    fontsize=10,
                )

        opts = {
            "alpha": 0.10,
            "scale": 1,
            "angles": "xy",
            "scale_units": "xy",
            "headlength": 0,
            "headaxislength": 0,
            "linewidth": 1.0,
            "rasterized": True,
        }
        ax.quiver(x0[:, 0], x0[:, 1], (x1 - x0)[:, 0], (x1 - x0)[:, 1], **{"color": self.linecolors[0], **opts})
        ax.quiver(x1[:, 0], x1[:, 1], (x2 - x1)[:, 0], (x2 - x1)[:, 1], **{"color": self.linecolors[1], **opts})
        ax.quiver(x2[:, 0], x2[:, 1], (x3 - x2)[:, 0], (x3 - x2)[:, 1], **{"color": self.linecolors[2], **opts})
        ax.scatter(x3[:, 0], x3[:, 1], color=self.linecolors[0], s=1, rasterized=True, alpha=0.20)

        ax.plot(
            [x0[exemplar, 0], x1[exemplar, 0], x2[exemplar, 0]],
            [x0[exemplar, 1], x1[exemplar, 1], x2[exemplar, 1]],
            "-",
            color=exemplar_color,
            linewidth=2,
            zorder=4,
            rasterized=True,
        )

        if arrows:
            ax.annotate(
                s="",
                xy=(-0.125, -0.5),
                xytext=(-0.125, 0.5),
                arrowprops=dict(arrowstyle="<->, head_width=.1, head_length=.2", ec="black", lw="0.5"),
                zorder=2,
            )
            self.arcarrow(x0[exemplar, :], x1[exemplar, :])
            self.arcarrow(x1[exemplar, :], x2[exemplar, :])
            self.arcarrow(x2[exemplar, :], x3[exemplar, :])
            ax.text(-0.09, -0.60, r"$x_1$", ha="center", va="center", fontsize=10)
            ax.text(0.13, -0.38, r"$x_2$", ha="center", va="center", fontsize=10)
            ax.text(0.60, -0.40, r"$x_3$", ha="center", va="center", fontsize=10)
            ax.text(1.10, -0.44, r"$x_4$", ha="center", va="center", fontsize=10)
            ax.text(1.97, -0.27, r"$\mathbf{y}$", ha="center", va="center", fontsize=10)

        ax.arrow(
            x2[exemplar, 0],
            x2[exemplar, 1],
            x3[exemplar, 0] - x2[exemplar, 0],
            x3[exemplar, 1] - x2[exemplar, 1],
            color=exemplar_color,
            linewidth=2,
            head_width=0.05,
            head_length=0.04,
            overhang=0.1,
            length_includes_head=True,
            zorder=4,
        )
        ax.scatter(
            [
                x0[exemplar, 0],
            ],
            [
                x0[exemplar, 1],
            ],
            s=40,
            marker="s",
            linewidth=1,
            edgecolors="black",
            facecolors=exemplar_color,
            alpha=0.8,
            zorder=3,
            rasterized=True,
        )
        ax.scatter(
            [x0[exemplar, 0], x1[exemplar, 0], x2[exemplar, 0]],
            [x0[exemplar, 1], x1[exemplar, 1], x2[exemplar, 1]],
            s=20,
            linewidth=1,
            edgecolors="black",
            facecolors=exemplar_color,
            alpha=0.8,
            zorder=5,
            rasterized=True,
        )

        ax.set_xlim(-0.01, 1.8)
        ax.set_ylim(*self.rangey)
        ax.set_aspect("equal")

        ax.set_xticks([])
        ax.set_yticks([])

        return ax

## 1. Prior & forward simulator

The prior is a zero-mean Gaussian with different scales per parameter. The forward model computes the arm's end position by chaining the three segments.

In [ ]:
def prior():
    """4D Gaussian prior over (height, angle_1, angle_2, angle_3)."""
    scales = np.array([0.25, 0.5, 0.5, 0.5])
    return dict(parameters=np.random.normal(loc=0, scale=scales))


def observation_model(parameters):
    """Forward kinematics: arm configuration -> 2D end-effector position."""
    height, a1, a2, a3 = parameters
    l1, l2, l3 = 0.5, 0.5, 1.0
    x1 = l1*np.sin(a1) + l2*np.sin(a1+a2) + l3*np.sin(a1+a2+a3) + height
    x2 = l1*np.cos(a1) + l2*np.cos(a1+a2) + l3*np.cos(a1+a2+a3)
    return dict(observables=np.array([x1, x2]))


variable_names = ["height_arm", "angle_1", "angle_2", "angle_3"]
variable_names_nice = [" ".join(v.title().split("_")) for v in variable_names]

In [ ]:
simulator = bf.make_simulator([prior, observation_model])

training_data = simulator.sample(10_000)
print("observables:", training_data["observables"].shape,
      " parameters:", training_data["parameters"].shape)

In [ ]:
PINK = "#E7298A"

def plot_params_kinematic(params=None, params2=None):
    _, ax = plt.subplots(1, 4, sharex=True, sharey=True, layout="constrained", figsize=(10, 2))
    for i, (a, name) in enumerate(zip(ax, variable_names_nice)):
        if params2 is not None: a.hist(params2[:, i], density=True, color="black", alpha=.5)
        if params is not None:  a.hist(params[:, i], density=True, color=PINK)
        a.set_xlabel(name)
    ax[0].set_ylabel("Density"); ax[0].set_ylim(0, 1.6); ax[0].set_xlim(-4, 4)

def plot_arm_posterior(posterior_samples, obs, title="Arm configurations"):
    _, ax = plt.subplots(figsize=(3, 3))
    m = InverseKinematicsModel(linecolors=[[PINK], [PINK], [PINK]])
    m.update_plot_ax(ax, posterior_samples["parameters"][0],
                     obs["observables"][0, ::-1], exemplar_color=PINK)
    ax.set_title(title)

## 2. Build & train the diffusion model

The **adapter** just names the tensors (`parameters` → target, `observables` → conditions). The inference network is a `DiffusionModel`. We standardise the conditions.

In [ ]:
adapter = (
    bf.adapters.Adapter()
    .to_array()
    .convert_dtype("float64", "float32")
    .rename("parameters", "inference_variables")
    .rename("observables", "inference_conditions")
)

# TODO(1): build a BasicWorkflow with a DiffusionModel inference network.
#   - adapter=adapter, simulator=simulator
#   - inference_network=bf.networks.DiffusionModel()
#   - standardize="inference_conditions"
#   Docs: https://bayesflow.org/main/api/bayesflow.networks.html
workflow_dm = ...  # <-- replace this

# TODO(2): train offline for 200 epochs, batch_size 128 (~2 min).
history = ...  # <-- replace this

### Amortized inference for a new observation

In [ ]:
obs = {"observables": np.array([[0.0, 1.5]])}

posterior = workflow_dm.sample(conditions=obs, num_samples=1000)
plot_params_kinematic(posterior["parameters"][0], training_data["parameters"])
plot_arm_posterior(posterior, obs)

You should see a **bimodal** posterior — two families of arm configurations reach the same point. That multimodality is exactly what makes this problem interesting.

### Is it the *correct* posterior? Coverage check

One observation looks plausible, but we must validate systematically. **Coverage** checks whether credible intervals contain the truth at the nominal rate, across many datasets.

In [ ]:
test_data = simulator.sample(100)
post_test = workflow_dm.sample(conditions=test_data, num_samples=100)
f = bf.diagnostics.plots.coverage(
    estimates=post_test, targets=test_data, variable_names=variable_names_nice
)

## 3. What is a diffusion model? (in one screen)

**Forward process (training).** Take a clean parameter vector $\boldsymbol\theta_0$ and add noise over a continuous time $t \in [0,1]$:

$$\boldsymbol\theta_t = \alpha_t\,\boldsymbol\theta_0 + \sigma_t\,\boldsymbol\epsilon,\qquad \boldsymbol\epsilon \sim \mathcal{N}(0, I).$$

The network sees $(\boldsymbol\theta_t, \mathbf{x})$ and learns the **score** $\nabla_{\boldsymbol\theta_t} \log p(\boldsymbol\theta_t \mid \mathbf{x})$ — the direction that increases posterior density.

**Reverse process (sampling).** Start from noise and integrate a reverse SDE/ODE defined entirely by the noise schedule and the learned score, denoising back to a clean sample.

**Relatives** (all in BayesFlow, all reparameterisations of the same idea):
- **Flow matching** — directly predicts the deterministic reverse *velocity field*.
- **Consistency models** — learn to jump to a clean sample in very few steps (fast sampling).

> Deep dive & benchmarks: Arruda et al. (2025), *Diffusion Models in SBI: A Tutorial Review* — <https://arxiv.org/abs/2512.20685>

#### Watch the denoising (optional)

`stop_time` halts reverse diffusion early. At $t\approx 1$ samples are pure noise; at $t=0$ they are the full posterior. Sweep it to *see* the posterior condense.

In [ ]:
import logging; logging.getLogger("bayesflow").setLevel(logging.ERROR)
for stop_t in [0.8, 0.5, 0.2, 0.0]:
    part = workflow_dm.sample(conditions=obs, num_samples=300, stop_time=stop_t)
    plot_params_kinematic(part["parameters"][0])
    plt.suptitle(f"partially denoised posterior at stop_time = {stop_t}")

## 4. (Optional) Flow matching & consistency models

Swap only the inference network to compare parameterisations on the same problem. Trains two more models (~4 min); skip if you're short on time.

In [ ]:
workflow_fm = bf.BasicWorkflow(adapter=adapter, simulator=simulator,
    inference_network=bf.networks.FlowMatching(), standardize="inference_conditions")
workflow_cm = bf.BasicWorkflow(adapter=adapter, simulator=simulator,
    inference_network=bf.networks.StableConsistencyModel(), standardize="inference_conditions")

for w in (workflow_fm, workflow_cm):
    w.fit_offline(training_data, epochs=200, batch_size=128, verbose=0)

_, ax = plt.subplots(1, 3, figsize=(10, 4), subplot_kw=dict(box_aspect=0.9), layout="constrained")
for a, w, name, col in zip(ax, [workflow_dm, workflow_fm, workflow_cm],
                           ["Diffusion", "Flow Matching", "Consistency"],
                           ["#E7298A", "#1B9E77", "#E6AB02"]):
    ps = w.sample(conditions=obs, num_samples=300)
    InverseKinematicsModel(linecolors=[col]*3).update_plot_ax(
        a, ps["parameters"][0], obs["observables"][0, ::-1], exemplar_color="#e6e7eb")
    a.set_title(name)

## 5. ⭐ The main task: write your own guidance

The real superpower of score-based models: because the model exposes a **score**, we can **modify inference after training** by adding the gradient of a differentiable *preference* term to the reverse dynamics. This is the basis of compositional / constrained SBI.

### The guidance contract (read carefully)

You supply one or more **constraint functions** `c(z)` via `guidance_kwargs=dict(constraints=[...], guidance_strength=λ)`. BayesFlow then adds, at each reverse step, the gradient of

$$\log g(z) = -\,\mathrm{softplus}\big(s(t)\,c(z)\big)$$

to the score. The consequences you must design around:

1. **Sign convention:** the constraint is *satisfied* exactly when **`c(z) <= 0`**. The term pushes samples toward that region. Make `c(z)` large-positive when the preference is violated.
2. **Input space:** `c(z)` receives the raw **inference-variables** tensor, shape `(..., 4)` = `[height, angle_1, angle_2, angle_3]`. If you standardised `inference_variables` in the workflow, undo it first (helper line provided). Here we only standardised the *conditions*, so `z` is already in parameter space — but the guard makes your code robust either way.
3. **Differentiability:** use `keras.ops.*` so gradients flow on any backend.

### Your first constraint: elbow-up vs elbow-down

The posterior is bimodal in the first joint angle $\alpha_1$. We'll use its sign as a selector:
- **elbow-up** should prefer $\sin(\alpha_1) \ge 0$,
- **elbow-down** should prefer $\sin(\alpha_1) \le 0$.

Translate each *preference* into a `c(z) <= 0` condition.

In [ ]:
def elbow_constraint(workflow, target="elbow-up"):
    """Guidance constraint. Satisfied <=> c(z) <= 0.

    elbow-up   prefers sin(a1) >= 0
    elbow-down prefers sin(a1) <= 0
    """
    def c(z):
        std = workflow.approximator.standardize_layers
        if "inference_variables" in std:            # undo standardisation if present
            z = std["inference_variables"](z, forward=False)
        a1 = z[..., 1]                                # first joint angle
        # TODO(3): return c(z) that is <= 0 exactly when `target` is satisfied.
        #   Recall: elbow-up wants sin(a1) >= 0, elbow-down wants sin(a1) <= 0.
        #   Use keras.ops.sin so gradients flow. One expression, depends on `target`.
        raise NotImplementedError("Write the elbow constraint (TODO 3)")
    return c

Now steer sampling. Compare **unguided** vs **guided** posteriors side by side.

In [ ]:
# TODO(4): sample unguided and guided posteriors, then plot both.
#   Guided call:
#     workflow_dm.sample(conditions=obs, num_samples=300,
#         guidance_kwargs=dict(constraints=[elbow_constraint(workflow_dm, target="elbow-up")],
#                              guidance_strength=1.0))
#   Then draw both with InverseKinematicsModel(...).update_plot_ax(...) as above.
...  # <-- replace this

> 🔎 If it worked, the guided arms collapse onto the **elbow-up** family while the unguided posterior keeps both modes. Try `target="elbow-down"` and vary `guidance_strength` (0.1 → 5): weak guidance barely tilts the balance, strong guidance can over-constrain and distort the posterior. Guidance is a *soft* prior at inference time, not a hard filter.

## 6. ⭐⭐ Design your own constraint — and tame the scaling

The elbow constraint was easy on the sampler because $\sin(\alpha_1)$ is **bounded and periodic**: once $\sin(\alpha_1)\approx 1$ the gradient vanishes and the sample can't run away. Now try a **bounded-region** constraint that is *not* self-limiting:

> Keep the middle joint within a band: $|\alpha_2| \le 0.5$. As a single `c(z) <= 0` rule: `c(z) = |α₂| - 0.5`.

### ⚠️ The scaling trap
By default BayesFlow scales the constraint by $s(t) = \alpha_t^2/\sigma_t^2$ — the signal-to-noise ratio, which **explodes as $t\to 0$** (end of sampling). For a linear, unbounded constraint the guidance gradient then grows without bound and **diverges** the sampler (you'll see $|\alpha_2|$ blow up to 10s instead of shrinking!). The fix is to pass your own, **bounded** `scaling_function` inside `guidance_kwargs` — e.g. cap the SNR. This is the practical knob that makes constrained sampling stable.

For fully custom behaviour beyond additive constraints, you can also override `workflow.approximator.inference_network.guidance_function` directly (see its docstring).

In [ ]:
# TODO(5a): finish the bounded-region constraint: return |a2| - bound with keras.ops.
def bounded_angle2_constraint(workflow, bound=0.5):
    def c(z):
        std = workflow.approximator.standardize_layers
        if "inference_variables" in std:
            z = std["inference_variables"](z, forward=False)
        raise NotImplementedError("Return |a2| - bound (TODO 5a)")
    return c

# TODO(5b): first try guidance WITHOUT a scaling_function and observe |angle_2| diverge.
#           Then finish `capped_snr_scaling` below and pass it via guidance_kwargs to fix it.
def capped_snr_scaling(workflow, cap=10.0):
    ns = workflow.approximator.inference_network.noise_schedule
    def s(t):
        log_snr = ns.get_log_snr(t, training=False)
        alpha, sigma = ns.get_alpha_sigma(log_snr)
        snr = keras.ops.square(alpha) / keras.ops.square(sigma)
        # TODO(5b): cap `snr` at `cap` so the guidance term stays bounded near t=0.
        #   Hint: keras.ops.minimum(...)
        raise NotImplementedError("Cap the SNR (TODO 5b)")
    return s

# ...then sample with guidance_kwargs=dict(constraints=[...], guidance_strength=3.0,
#          scaling_function=capped_snr_scaling(workflow_dm)) and plot the effect.
...  # <-- replace this

## 🎉 Summary

1. **SBI** turns simulation into amortized posteriors — instant inference for new data.
2. **Diffusion models** learn a score and sample by reverse denoising; they excel at multimodal, high-dimensional posteriors.
3. **Guidance** exploits the score to steer sampling *after* training with a differentiable `c(z) <= 0` constraint — no retraining required.

**Further reading & next steps**
- BayesFlow: <https://bayesflow.org> (summary networks for high-dim data, SBC, coverage).
- Diffusion-SBI tutorial review: <https://bayesflow-org.github.io/diffusion-experiments/>
- Compositional / hierarchical SBI with guidance: <https://arxiv.org/abs/2505.14429>

> 📎 **TODO (course):** add the lecture link on score-based generative models & classifier-free guidance, plus any assignment submission instructions, here.